# Analysis — Tweede Kamer AI documents

In [ ]:
import ast
import re
from collections import Counter, defaultdict
from math import ceil

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

In [ ]:
tk = pd.read_csv('tweede_kamer_data.csv', index_col=0)


In [ ]:
tk.head()

In [ ]:
tk['year'] = tk['year'].astype('Int64')

# De-duplicate keywords within each document
tk['matched_keywords_all'] = tk['matched_keywords_all'].apply(
    lambda v: list(dict.fromkeys(ast.literal_eval(v))) if isinstance(v, str) else list(dict.fromkeys(v or []))
)

In [ ]:
print(tk['type'].value_counts().to_string())
print(f"\nYear range: {tk['year'].min()} – {tk['year'].max()}")

In [ ]:
tk['relevant_len_words'] = tk['body'].str.count(r'\b\w+\b')

In [ ]:
_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord", "Twitter", "X",
]
_COMPANY_NAMES_LOWER = {n.lower() for n in _COMPANY_NAMES}

_COMPANY_PATTERNS = {
    name: re.compile(rf'\b{re.escape(name)}\b', re.IGNORECASE)
    for name in _COMPANY_NAMES
}

def company_counts_exact(text):
    if not isinstance(text, str):
        return Counter()
    return Counter({
        name: len(pat.findall(text))
        for name, pat in _COMPANY_PATTERNS.items()
        if pat.search(text)
    })

tk['company_hits']   = tk['body'].apply(company_counts_exact)
tk['n_company_hits'] = tk['company_hits'].apply(lambda x: sum(x.values()))

In [ ]:
## Top mentioned companies

total_counts = Counter()
for hits in tk['company_hits']:
    total_counts.update(hits)

top_n = 20
company_df = (
    pd.DataFrame(total_counts.items(), columns=['Company', 'Mentions'])
    .sort_values('Mentions', ascending=False)
    .head(top_n)
    .sort_values('Mentions')          # ascending so largest is at the top of barh
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(company_df['Company'], company_df['Mentions'], color='#4472C4', edgecolor='white')
ax.set_xlabel('Number of mentions')
ax.set_title(f'Top {top_n} most mentioned companies — Tweede Kamer')
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Trends over time

In [ ]:
tk['keywords_parsed'] = tk['matched_keywords_all'].apply(
    lambda v: ast.literal_eval(v) if isinstance(v, str) else (v or [])
)
tk['ai_keywords'] = tk['keywords_parsed'].apply(
    lambda kws: [k for k in kws if k.lower() not in _COMPANY_NAMES_LOWER]
)
tk['n_ai_keywords']    = tk['ai_keywords'].apply(len)
tk['keyword_density']  = tk['n_ai_keywords'] / tk['relevant_len_words']

tk_year = tk.dropna(subset=['year']).copy()
tk_year['year'] = tk_year['year'].astype(int)

In [ ]:
yearly = tk_year.groupby('year').size().reset_index(name='doc_count')

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(yearly['year'], yearly['doc_count'], width=0.6, color='#4472C4', edgecolor='white')
ax.set_xlabel('Year')
ax.set_ylabel('Number of documents')
ax.set_title('AI-related Tweede Kamer documents per year')
ax.set_xticks(yearly['year'])
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Keyword frequency

In [ ]:
all_keywords = [kw for kws in tk['ai_keywords'] for kw in kws]
kw_counts = Counter(all_keywords)
kw_df = (
    pd.DataFrame(kw_counts.items(), columns=['keyword', 'count'])
    .sort_values('count', ascending=False)
)

top_n = 20
plot_df = kw_df.head(top_n).sort_values('count')   # ascending so largest is at the top

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df['keyword'], plot_df['count'], color='#4472C4', edgecolor='white')
ax.set_xlabel('Total document occurrences')
ax.set_title(f'Top {top_n} AI keywords — Tweede Kamer')
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
top_keywords = [k for k, _ in kw_counts.most_common(8)]

year_kw = defaultdict(Counter)
for _, row in tk_year.iterrows():
    year_kw[int(row['year'])].update(row['ai_keywords'])

years = sorted(year_kw.keys())
doc_counts_by_year = tk_year.groupby('year').size()

n_topics = len(top_keywords)
cmap   = plt.get_cmap('tab10', n_topics)
colors = cmap(np.arange(n_topics))

fig, ax = plt.subplots(figsize=(12, 6))
for color, kw in zip(colors, top_keywords):
    vals = [year_kw[y][kw] / doc_counts_by_year[y] for y in years]
    ax.plot(years, vals, marker='o', label=kw, color=color,
            markerfacecolor='white', markeredgewidth=2, linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Avg mentions per document')
ax.set_title('Top AI keywords over time — Tweede Kamer')
ax.set_xticks(years)
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()